# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.
2. **Chạy trực tiếp (Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang.

In [ ]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [ ]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os

# Luôn luôn xuất phát từ thư mục gốc /content (chống lỗi clone lồng nhau ZAIC_2026/ZAIC_2026)
%cd /content

if not os.path.exists('/content/ZAIC_2026/.git'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git /content/ZAIC_2026

%cd /content/ZAIC_2026
!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

In [ ]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Tự động giải nén lên ổ cứng cục bộ /content)
import os, shutil, sys
from pathlib import Path

%cd /content/ZAIC_2026

# Kiểm tra xem /content/data có sẵn chưa
has_local_data = os.path.exists('/content/data/train/train.jsonl') and os.path.exists('/content/data/train/zaic_tools/apps/banking.py')

if not has_local_data:
    # Tìm file zip data.zip trên Drive hoặc /content
    zip_candidates = ['/content/drive/MyDrive/data.zip', '/content/data.zip', '/content/drive/MyDrive/ZAIC_2026/data.zip']
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if found_zip:
        print(f'📦 Đang giải nén dữ liệu từ {found_zip} sang ổ cứng cục bộ /content/ ... (mất ~15 giây)')
        !unzip -q -o "{found_zip}" -d /content/
    else:
        print('⚠️ Không tìm thấy data.zip trên Drive hoặc /content!')

# Tạo liên kết data/train và data/public_test trong repo
os.makedirs('data', exist_ok=True)
if os.path.exists('/content/data/train'):
    !rm -rf data/train
    !ln -s /content/data/train data/train
if os.path.exists('/content/data/public_test'):
    !rm -rf data/public_test
    !ln -s /content/data/public_test data/public_test

# KIỂM TRA TỰ ĐỘNG: Import thử zaic_tools để đảm bảo 100% không còn lỗi ModuleNotFoundError
try:
    if 'data/train' not in sys.path:
        sys.path.insert(0, 'data/train')
    from zaic_tools import ToolEnv
    print('✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!')
    !ls -lh data/train | head -n 6
except Exception as e:
    print('❌ LỖI DỮ LIỆU:', e)
    print('Vui lòng kiểm tra file data.zip có chứa thư mục zaic_tools/apps hay không!')

### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [ ]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
%cd /content/ZAIC_2026
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.pipeline' || true
import os, torch, time
time.sleep(2)
os.makedirs('runs', exist_ok=True)

vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
gpu_mem = 0.88 if vram < 20 else 0.90
print(f'GPU VRAM: {vram:.1f} GB -> Cấu hình --gpu-mem {gpu_mem}')

# Chạy ngầm bằng nohup: không bị ngắt khi gập máy tính
!nohup python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem {gpu_mem} \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v4_guarded.jsonl > runs/dev_v4_guarded.nohup.log 2>&1 &

print('🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')

In [ ]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# %cd /content/ZAIC_2026
# import os, torch
# os.makedirs('runs', exist_ok=True)
# vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
# gpu_mem = 0.88 if vram < 20 else 0.90
#
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem {gpu_mem} \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v4_guarded.jsonl

In [ ]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Chạy cell này bất cứ lúc nào để kiểm tra)
%cd /content/ZAIC_2026
import time, os
from IPython.display import clear_output

out_path = 'runs/dev_v4_guarded.jsonl'
log_path = 'runs/dev_v4_guarded.jsonl.log'
nohup_log = 'runs/dev_v4_guarded.nohup.log'

for _ in range(60):  # cập nhật mỗi 5s trong 5 phút
    clear_output(wait=True)
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 60)
    if os.path.exists(log_path):
        !tail -n 12 runs/dev_v4_guarded.jsonl.log
    elif os.path.exists(nohup_log):
        !tail -n 8 runs/dev_v4_guarded.nohup.log
    
    is_running = os.popen("pgrep -f 'lifemate.pipeline'").read().strip()
    if is_running:
        print(f'\n⚡ Tiến trình GPU đang chạy (PID: {is_running})...')
    else:
        if done_count > 0:
            print('\n🎉 TIẾN TRÌNH ĐÃ HOÀN THÀNH XONG!')
        else:
            print('\n⏹️ Tiến trình chưa chạy hoặc bị lỗi khởi động. Xem log ở cell dưới.')
        break
    time.sleep(5)

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
%cd /content/ZAIC_2026
!python eval/scorer.py --sub runs/dev_v4_guarded.jsonl

In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
%cd /content/ZAIC_2026
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')